# AIC 2026 Retrieval Demo trên Kaggle

Trước khi Run all, chọn **Add Input** và gắn ít nhất: `clip-features-32-aic25-b1`, `map-keyframes-aic25-b1`, `dataset-ai-challenge-keyframe`. Khuyến nghị gắn thêm `media-info-aic25-b1` và `objects-aic25-b1-zip`.

Notebook này chỉ `git clone`/`git pull` source code. Nó không gọi API tải/copy AIC dataset. Bật Internet lần đầu để cài dependency và tải checkpoint CLIP ViT-B/32 nếu checkpoint chưa được cache.

In [ ]:
%reset -f

In [ ]:
from pathlib import Path

REPO = Path('/kaggle/working/HCMAIC')
if not REPO.exists():
    !git clone https://github.com/DangMinhSang/HCMAIC.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log -1 --oneline


In [ ]:
# Không tải dataset: cell này chỉ cài code dependency nhỏ và CLIP package.
!python -m pip install -q -r Code/requirements.txt


In [ ]:
import os
import sys

os.environ['AIC_DATA_ROOT'] = '/kaggle/input'  # Chỉ đọc Kaggle Inputs đã gắn
sys.path.insert(0, str(REPO / 'Code'))

from data_paths import AICPaths
paths = AICPaths.from_environment()
print('CLIP features:', paths.features_dir)
print('Mapping:', paths.mapping_dir)
print('Keyframe roots:', len(paths.keyframe_roots))
print('Metadata:', paths.metadata_dir or 'không gắn')
print('Objects:', paths.objects_dir or 'không gắn')


## OCR index (tùy chọn, chạy một lần)

Bật `BUILD_OCR_INDEX = True` để đọc keyframe đã mount và lưu **text-only index** vào `/kaggle/working`. Lần sau dashboard nạp index này vào RAM; truy vấn text trên biển báo không OCR lại ảnh nên rất nhanh. Không có AIC image/video/feature nào được copy hay tải về.

In [ ]:
from pathlib import Path

OCR_INDEX = Path('/kaggle/working/aic_ocr_index.jsonl.gz')
BUILD_OCR_INDEX = False  # Đổi thành True khi muốn pre-OCR toàn bộ keyframe.
if BUILD_OCR_INDEX:
    !python -m pip install -q -r Code/requirements-ocr.txt
    !python Code/build_ocr_index.py --output {OCR_INDEX}

os.environ['AIC_OCR_INDEX'] = str(OCR_INDEX)
os.environ['AIC_PRELOAD_FEATURES'] = '1'  # Giữ CLIP feature đã chuẩn hóa trong RAM để query lặp nhanh.
print('OCR index:', OCR_INDEX if OCR_INDEX.exists() else 'chưa build — CLIP-only')


In [ ]:
# Xóa module cũ khỏi kernel sau mỗi git pull; `%reset` chỉ xóa biến, không xóa sys.modules.
for _module in ('dashboard', 'share_dashboard', 'retrieval', 'ocr_index', 'data_paths', 'clip_encoder', 'qa', 'query_language'):
    sys.modules.pop(_module, None)
from share_dashboard import launch_dashboard

# Gradio chỉ tạo tunnel cho Kaggle; UI chính là dashboard keyframe-card tại URL bên dưới.
dashboard_url = launch_dashboard(share=True)
print('Open dashboard:', dashboard_url)
